# Importing Data — CSV, JSON, Excel, and SQL

## Introduction

Real-world data arrives in many formats. Pandas can read almost all of them with a consistent API. This notebook covers the most common: CSV, JSON, and Excel files, plus connecting to SQL databases. We'll also cover the key parameters that control how data is parsed and what to check right after loading.

## Objectives

You will be able to:

* Load CSV, JSON, and Excel files into DataFrames
* Use key `read_csv` parameters: `index_col`, `parse_dates`, `dtype`, `usecols`, `nrows`
* Handle common load-time issues: missing values, encoding, separators
* Export DataFrames back to file formats
* Inspect a freshly loaded DataFrame for data quality issues

---

## Reading CSV Files

In [ ]:
import pandas as pd
import numpy as np
import io

# Simulate a CSV file in memory for this demo
csv_data = """date,product,qty,price,store
2024-01-05,Widget,10,9.99,NYC
2024-01-06,Gadget,,24.99,LA
2024-01-07,Widget,5,9.99,NYC
2024-01-08,Doohickey,3,14.50,Chicago
2024-01-09,Gadget,8,24.99,LA
"""

df = pd.read_csv(io.StringIO(csv_data))
print(df)
print(df.dtypes)

In [ ]:
# Key parameters for read_csv
df = pd.read_csv(
    io.StringIO(csv_data),
    parse_dates=['date'],       # parse 'date' column as datetime
    dtype={'price': float},     # specify column types explicitly
    usecols=['date','product','qty','price'],  # only load these columns
)
print(df)
print(df.dtypes)

In [ ]:
# nrows — load only first N rows (useful for exploring large files)
df_preview = pd.read_csv(io.StringIO(csv_data), nrows=3)
print(df_preview)

# skiprows — skip rows at the top (if file has a preamble/header description)
# pd.read_csv('file.csv', skiprows=3)

In [ ]:
# Handling missing values
# By default Pandas treats '', 'NA', 'NaN', 'N/A', 'null', 'None' as NaN
# To add your own sentinel values:
df_na = pd.read_csv(io.StringIO(csv_data), na_values=['-', '?', 'missing'])
print(df_na.isnull().sum())  # count NaN per column

In [ ]:
# Real file would be:
# df = pd.read_csv('data/sales.csv', parse_dates=['date'], index_col='date')

# Other separators
tsv_data = "a\tb\tc\n1\t2\t3\n4\t5\t6"
df_tsv = pd.read_csv(io.StringIO(tsv_data), sep='\t')
print(df_tsv)

# Semicolons (common in European locale CSVs where comma is the decimal separator)
# pd.read_csv('euro_data.csv', sep=';', decimal=',')

---

## Reading JSON

In [ ]:
import json

# JSON as array of objects (most common API response format)
json_str = """
[
    {"id": 1, "name": "Alice", "score": 88.5},
    {"id": 2, "name": "Bob",   "score": 92.0},
    {"id": 3, "name": "Carol", "score": 75.5}
]
"""
df_json = pd.read_json(io.StringIO(json_str))
print(df_json)

In [ ]:
# JSON with nested structure — use json_normalize
nested_json = [
    {'id': 1, 'name': 'Alice', 'address': {'city': 'NYC', 'state': 'NY'}},
    {'id': 2, 'name': 'Bob',   'address': {'city': 'LA',  'state': 'CA'}},
]

df_nested = pd.json_normalize(nested_json, sep='_')
print(df_nested)
# address.city and address.state become address_city, address_state

---

## Reading Excel

In [ ]:
# Requires openpyxl: pip install openpyxl

# Basic read
# df_xl = pd.read_excel('data/report.xlsx')

# Specify sheet name and header row
# df_xl = pd.read_excel(
#     'data/report.xlsx',
#     sheet_name='Sales 2024',
#     header=2,        # row index of the header row
#     usecols='A:F',   # Excel column range
# )

# Read all sheets at once — returns dict of DataFrames
# sheets = pd.read_excel('data/report.xlsx', sheet_name=None)
# for name, df in sheets.items():
#     print(f"Sheet '{name}': {df.shape}")

print("Excel import requires: pip install openpyxl")
print("Usage: pd.read_excel('file.xlsx', sheet_name='Sheet1')")

---

## Reading from SQL Databases

In [ ]:
import sqlite3

# Create a demo in-memory SQLite database
conn = sqlite3.connect(':memory:')
conn.execute("""CREATE TABLE sales (id INT, product TEXT, amount REAL, date TEXT)""")
conn.executemany("INSERT INTO sales VALUES (?,?,?,?)", [
    (1, 'Widget', 99.99, '2024-01-01'),
    (2, 'Gadget', 249.99, '2024-01-02'),
    (3, 'Widget', 99.99, '2024-01-03'),
])
conn.commit()

# Read with pd.read_sql
df_sql = pd.read_sql("SELECT * FROM sales WHERE amount > 100", conn)
print(df_sql)

# Full query power
df_agg = pd.read_sql(
    "SELECT product, COUNT(*) as count, SUM(amount) as total FROM sales GROUP BY product",
    conn
)
print(df_agg)

conn.close()

---

## Exporting DataFrames

In [ ]:
df = pd.DataFrame({'a': [1, 2, 3], 'b': ['x', 'y', 'z']})

# CSV
df.to_csv('output.csv', index=False)   # index=False avoids writing row numbers

# JSON
df.to_json('output.json', orient='records', indent=2)

# Excel (requires openpyxl)
# df.to_excel('output.xlsx', sheet_name='Data', index=False)

# Verify round-trip
df_back = pd.read_csv('output.csv')
print(df_back)

import os
os.remove('output.csv')
os.remove('output.json')

---

## Post-Load Checklist

In [ ]:
# Always run these after loading a new dataset
def quick_check(df, name='DataFrame'):
    print(f"=== {name} ===")
    print(f"Shape:    {df.shape}")
    print(f"\nColumn types:")
    print(df.dtypes)
    print(f"\nMissing values:")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.any() else "None")
    print(f"\nPreview:")
    print(df.head(3))
    print(f"\nStatistics:")
    print(df.describe())

# Demo on a small dataset
demo = pd.read_csv(io.StringIO(csv_data), parse_dates=['date'])
quick_check(demo, 'Sales Data')

---

## Practice

In [ ]:
# Use this CSV string for the exercises
raw = """student_id,name,math,science,english,grade_date
101,Alice,92,88,79,2024-06-01
102,Bob,75,,85,2024-06-01
103,Carol,88,91,92,2024-06-01
104,Dan,60,72,68,2024-06-01
105,Eve,95,97,91,2024-06-01
"""

# 1. Load the CSV with student_id as index and grade_date parsed as a date
df_students = None
print(df_students)

In [ ]:
# 2. How many missing values are there, and in which column?
# 3. What is the average math score?
# 4. Export only students with math >= 85 to a CSV called 'top_math.csv'

## Summary

| Format | Read | Write |
|--------|------|-------|
| CSV | `pd.read_csv(path)` | `df.to_csv(path, index=False)` |
| JSON | `pd.read_json(path)` | `df.to_json(path, orient='records')` |
| Excel | `pd.read_excel(path, sheet_name=)` | `df.to_excel(path, index=False)` |
| SQL | `pd.read_sql(query, conn)` | `df.to_sql(name, conn)` |

Key `read_csv` parameters to remember: `parse_dates`, `index_col`, `usecols`, `nrows`, `dtype`, `na_values`, `sep`. After loading any dataset, always check `.shape`, `.dtypes`, `.isnull().sum()`, and `.describe()`.